# Canonical Hashing and Graph Comparison — User Guide

Two graphs can represent the exact same information while disagreeing on blank node labels — labels are arbitrary identifiers, not part of the data. `isomorphic()` (`starlayergraph.compare`) checks graph equivalence correctly under that rule, RDF-1.2 triple terms included. `rdfc10_hash()` and `to_canonical_nquads()` (`starlayergraph.rdfc`) implement the [RDFC-1.0](https://www.w3.org/TR/rdf-canon/) canonicalization algorithm: a deterministic, blank-node-label-independent hash/serialization of a graph, so two isomorphic graphs always hash identically regardless of how their blank nodes happen to be labeled.

This is a general graph feature, not RDF-1.2-specific, but it lives in the same `starlayergraph` package - see the [Graphs guide](02-graphs.ipynb) for the triple-term/reification/literal semantics of the graphs being compared here. For a worked, end-to-end use of this (committing a hash of exactly the part of a graph a shape covers, then re-verifying it later), see the [subgraph extraction guide](04f-shacl-subgraph-extraction.ipynb)'s final section.

## How to run this notebook

See [Getting Started](01-getting-started.ipynb) if you haven't installed StarLayer yet. Run cells from top to bottom — later cells reuse variables from earlier ones.

In [1]:
from starlayer import StarLayerGraph, Namespace, Literal, BNode
from starlayergraph.compare import isomorphic
from starlayergraph.rdfc import rdfc10_hash, to_canonical_nquads

EX = Namespace("http://example.org/")

## `isomorphic()` and `rdfc10_hash()`

In [2]:
g1 = StarLayerGraph()
g1.bind("ex", EX)
b1 = BNode()
g1.add((EX.alice, EX.knows, b1))
g1.add((b1, EX.name, Literal("someone")))

# same shape, deliberately different (fresh, unrelated) blank node label
g2 = StarLayerGraph()
g2.bind("ex", EX)
b2 = BNode()
g2.add((EX.alice, EX.knows, b2))
g2.add((b2, EX.name, Literal("someone")))

print("isomorphic despite different blank node labels:", isomorphic(g1, g2))
print("hash g1:", rdfc10_hash(g1))
print("hash g2:", rdfc10_hash(g2))
print("hashes equal:", rdfc10_hash(g1) == rdfc10_hash(g2))

g3 = StarLayerGraph()
g3.bind("ex", EX)
g3.add((EX.alice, EX.knows, EX.bob))

print()
print("isomorphic to a graph with genuinely different data:", isomorphic(g1, g3))

isomorphic despite different blank node labels: True
hash g1: 236939de1885b1e84eef000f98a0803aeacd2611a57fe3cdac66ae4b9cc3cc1c
hash g2: 236939de1885b1e84eef000f98a0803aeacd2611a57fe3cdac66ae4b9cc3cc1c
hashes equal: True

isomorphic to a graph with genuinely different data: False


In [3]:
# to_canonical_nquads() is the canonical serialization the hash is derived from -
# blank nodes get deterministic c14n labels instead of their original arbitrary ones.
print(to_canonical_nquads(g1))

<http://example.org/alice> <http://example.org/knows> _:c14n0 .
_:c14n0 <http://example.org/name> "someone" .



This combination — extract exactly the part of a graph that matters, hash it canonically, and later re-verify the hash still matches — is the basis of the "commit a shape-scoped subgraph, detect if it later changes" workflow in the [subgraph extraction guide](04f-shacl-subgraph-extraction.ipynb)'s final section. That guide builds `commit_subgraph()`/`verify_commitment()` helpers on top of exactly the two functions shown here (`rdfc10_hash()` for the hash, `isomorphic()` for comparing extractions before/after a change).

## Further work

- **Highly symmetric blank-node graphs can be expensive to canonicalize.** RDFC-1.0 is a genuinely hard problem in the worst case; `rdfc10_hash()`/`to_canonical_nquads()` take a `permutation_budget` (default 100000) and raise `CanonicalizationComplexityError` rather than hanging indefinitely once exceeded — the spec's own required defense against "Dataset Poisoning" (RDFC-1.0 §7.1). Confirmed live: without this guard, the official test suite's own poisoning fixture (a 10-node blank-node clique) hangs rather than raising anything.